# Dictionary Utility Notebook

### 1. Introduction

This notebook provides a practical guide to the `africanlanguages.dictionary` module and demonstrates the main lookup operations:

- **Exact match lookup:** Retrieves an entry when the headword is known.
- **Fuzzy lookup:** Handle misspellings or phonetic variants in African-language queries.
- **Reverse lookup:** Search English terms across definitions and translations to find the corresponding headwords.
- **Sentence lookup:** Performs token-level lookup over full sentences.

Each section includes examples that illustrate how the search interface behaves in typical use cases.

### 2. Setup
Import the required classes and initialize the dictionary for the target language.  
This loads the entries, builds the internal index, and prepares the lookup interface.


In [1]:
from africanlanguages.dictionary import Dictionary

LANG = "yor"  # Yoruba dictionary

yoruba_dict = Dictionary(LANG)

print(f"Loaded Dictionary for: {yoruba_dict.language_code}")
print(f"Total entries: {len(yoruba_dict)}")

/home/adeleyi/projects/africanlanguages/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Dictionary for: yor
Total entries: 12226


### 2.1. Standard Lookup (Exact Match)
Exact lookup retrieves an entry only when the headword matches exactly as stored in the dictionary.
Tone marks and diacritics may affect results depending on how the word is stored.

In [2]:
word = "aja"
results_exact = yoruba_dict.lookup(word)

if results_exact:
    print(f"Matches returned: {len(results_exact)}")
    first = results_exact[0]
    print(f"Word: {first.word} | {first.definition}")
else:
    print("No entry found.")

Matches returned: 2
Word: aja | a dog


### 2.2. Fuzzy Lookup (African Language → English)
Fuzzy lookup is useful for handling misspellings, phonetic approximations, or uncertain spellings.
The `threshold` controls sensitivity, and `top_n` selects how many results to return.


In [3]:
word_fuzzy = "atewo"

results_fuzzy = yoruba_dict.lookup(word_fuzzy, exact_match=False, threshold=0.6, top_n=3)

print(f"Top 3 Fuzzy Matches for '{word_fuzzy}'")

if results_fuzzy:
    for i, entry in enumerate(results_fuzzy, 1):
        print(f"{i}. {entry.word} | {entry.definition}")
else:
    print("No fuzzy matches found.")

Top 3 Fuzzy Matches for 'atewo'
1. atàwo̩ | a dealer in hides, leather
2. bawo | how? in what way?
3. ṣawo | to be initiated into a secret


### 2.3. Reverse Lookup (English → African Language)
Reverse lookup checks the English query against the definition field.
It uses fuzzy matching internally, so it does not require exact phrasing.

In [4]:
english_word = "book"

results_reverse = yoruba_dict.lookup(english_word, exact_match=False, top_n=2)

print(f"Reverse Lookup for '{english_word}'")

if results_reverse:
    for entry in results_reverse:
        print(f"{entry.word} | {entry.definition}")
else:
    print("No matching entries found.")

Reverse Lookup for 'book'
iwe | book; paper
kíkọ | written; writing (book)


In [5]:
# Example 2
english_word = "diviner"

results_reverse = yoruba_dict.lookup(english_word, exact_match=False)

if results_reverse:
    print(f"English Word: {english_word} | {results_reverse}")

English Word: diviner | [DictionaryEntry(word='aláfọṣé', language='yor', part_of_speech='noun', definition='a diviner; one who deals with familiar spirits', examples=[], translations=[]), DictionaryEntry(word='abáwin-gbìmo', language='yor', part_of_speech='noun', definition='a spiritualist medium; diviner', examples=[], translations=[])]


### 3. Sentence Lookup

`lookup_sentence` performs token-level lookup across the entire sentence.
By default (`simple=True`), it returns a dictionary where each token maps to a list of definition strings.


In [6]:
# Yor --> Eng
sentence = "Mú aga náà wa."

print(f"Sentence: {sentence}")
results_simple = yoruba_dict.lookup_sentence(sentence)

print("- Sentence Lookup (simple mode) -")
print(results_simple)

Sentence: Mú aga náà wa.
- Sentence Lookup (simple mode) -
{'Mú': ['to take; bring; hold; fetch; seize; catch; arrest; twinge, implicate', 'sharp; acute; keen; sagacious; animated; tart; pungent', 'rice'], 'aga': ['a chair, a stool'], 'náà': [], 'wa.': []}


In [7]:
# Reverse sentence lookup
sentence = "I'm going to the market tomorrow."

print(f"Sentence: {sentence}\n")

results_simple = yoruba_dict.lookup_sentence(sentence, exact_match=False)
print("--- Sentence Lookup (simple mode) ---")
print(results_simple)

Sentence: I'm going to the market tomorrow.

--- Sentence Lookup (simple mode) ---
{"I'm": ['imọ', 'imu'], 'going': ['alò', 'ìlọ'], 'to': ['i', 'ṣ'], 'the': ['a', 'ọ'], 'market': ['ọjà', 'aítà'], 'tomorrow.': ['ọtunla', 'lọla (ni-ọla)']}


#### 3.2. Sentence Lookup (`simple=False`)
`simple=False` returns full `DictionaryEntry` objects for each token.  
This is useful when you need access to metadata such as part of speech, example sentences, or info present in the dictionary.


In [8]:
sentence = "I'm going to school"

results_detailed = yoruba_dict.lookup_sentence(sentence, exact_match=False, simple=False)

print("--- Sentence Lookup (detailed mode) ---")

print(results_detailed)

--- Sentence Lookup (detailed mode) ---
{"I'm": [DictionaryEntry(word='imọ', language='yor', part_of_speech='noun', definition='knowledge; opinion', examples=[], translations=[]), DictionaryEntry(word='imu', language='yor', part_of_speech='noun', definition='the nose; nostrils', examples=[], translations=[])], 'going': [DictionaryEntry(word='alò', language='yor', part_of_speech='noun', definition='goings; departure', examples=[], translations=[]), DictionaryEntry(word='ìlọ', language='yor', part_of_speech='noun', definition='departure; a going', examples=[], translations=[])], 'to': [DictionaryEntry(word='i', language='yor', part_of_speech='verb', definition='bá, to hide', examples=[], translations=[]), DictionaryEntry(word='ṣ', language='yor', part_of_speech='verb', definition='to fade; to be sterile', examples=[], translations=[])], 'school': [DictionaryEntry(word='iléwẹ', language='yor', part_of_speech='noun', definition='school-house', examples=[], translations=[]), DictionaryEnt

### 4.1 Troubleshooting

#### 1. No results for an African-language query
If an exact lookup returns no matches:
- Verify tone marks and diacritics.
- Try switching to `exact_match=False` to allow fuzzy comparisons.
- Lower the threshold if the match is phonetically distant.

#### 2. Too many fuzzy matches
- Increase the threshold (e.g., from 0.6 to 0.8).
- Reduce `top_n`.

#### 3. Sentence lookup misses some tokens
- The tokenization is simple and may not catch contracted or punctuation-bound forms exactly.
- Inspect the tokens manually if results look sparse.
- Use detailed mode (`simple=False`) for transparency.